# Seguridad ciudadana en el Perú mediante Machine Learning

Objetivo: reproducir validación, integración, EDA, clustering, clasificación, XAI e interpretación ética usando datos oficiales.

## Fuentes y metodología

Las fuentes están en `data/raw`. El nivel común validado es departamento-año para 2018-2024. Antes de modelar se inspeccionan columnas, nulos, duplicados y granularidad.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
SEED = 42
ROOT

WindowsPath('C:/Users/Iriarte 06/OneDrive/Documentos/ChatGPT/Parcial Criminalidad/criminalidad-peru-ml')

## Validación e integración

Se ejecuta el pipeline de procesamiento para construir el panel reproducible.

In [2]:
from data_processing import build_department_year_dataset
panel = build_department_year_dataset()
panel.shape, panel.head()

((175, 38),
   departamento  anio  denuncias_total  meses_observados  anio_completo  \
 0     AMAZONAS  2018             9193                12           True   
 1     AMAZONAS  2019             9927                12           True   
 2     AMAZONAS  2020             7917                12           True   
 3     AMAZONAS  2021             8679                12           True   
 4     AMAZONAS  2022             9057                12           True   
 
    denuncias_estafa  denuncias_extorsion  denuncias_hurto  denuncias_otros  \
 0             116.0                 23.0           1903.0           4238.0   
 1             124.0                 13.0           2090.0           4149.0   
 2             123.0                 16.0           1584.0           3438.0   
 3             233.0                 31.0           1831.0           3618.0   
 4             312.0                 54.0           1858.0           3824.0   
 
    denuncias_robo  ...  victimizacion_estafa_pct  victimiza

## EDA

Se generan tablas y figuras descriptivas. Cada figura se guarda en `outputs/figures`.

In [3]:
from modeling import setup, run_eda, run_clustering, run_classification
setup()
eda = run_eda(panel)
pd.DataFrame(eda['by_year'])

,anio,denuncias_total,denuncias_tasa_100k,victimizacion_pct,percepcion_inseguridad_pct,confianza_pnp_pct
0,2018,774804,2233.724765,23.636,50.504,16.592
1,2019,854898,2504.610722,23.758,49.986,18.524
2,2020,649169,1980.121157,20.240,24.788,26.650
3,2021,754638,2294.709367,15.486,46.854,21.776
4,2022,877736,2532.970945,20.078,54.164,19.680
5,2023,1044843,2823.427038,22.832,51.414,19.062
6,2024,1014315,2723.833548,24.440,50.866,17.772


Interpretación: revise la evolución anual para distinguir denuncias registradas, victimización, percepción y confianza. Estos conceptos no son equivalentes.

## Clustering

Se comparan K-Means y Agglomerative Clustering con variables estandarizadas.

In [4]:
clustering = run_clustering(panel)
clustering['best_model'], clustering['k']

('KMeans', 2)

In [5]:
pd.read_csv(ROOT / 'outputs' / 'tables' / 'cluster_profiles.csv')

,cluster,observaciones,territorios,denuncias_tasa_100k,victimization,percepcion,confianza,brecha
0,0,96,"AMAZONAS, ANCASH, APURIMAC, AREQUIPA, AYACUCHO...",2038.242808,21.006250,38.573958,20.197396,17.567708
1,1,79,"AREQUIPA, AYACUCHO, CAJAMARCA, CALLAO, CUSCO, ...",2932.451000,22.090506,57.105063,19.777848,35.014557


Interpretación: los clusters son perfiles exploratorios de patrones agregados; no son etiquetas de peligrosidad.

## Clasificación

Se predice victimización alta relativa en t usando variables históricas y rezagos. Se evita leakage temporal.

In [6]:
classification = run_classification(panel)
pd.read_csv(ROOT / 'outputs' / 'tables' / 'classification_benchmark.csv')

,modelo,accuracy,precision,recall,f1,roc_auc,pr_auc,cv_best_score
0,DummyClassifier,0.72,0.000000,0.0,0.000000,0.500000,0.280000,NaN
1,LogisticRegression,0.78,0.560000,1.0,0.717949,0.944444,0.760477,0.686694
2,RandomForest,0.92,0.777778,1.0,0.875000,0.974206,0.933005,0.731990


Interpretación: compare contra DummyClassifier. Accuracy no basta si el baseline no recupera la clase positiva.

## XAI, ética y conclusiones

La importancia por permutación indica relevancia predictiva, no causalidad. Deben considerarse sesgos de reporte, cobertura, medición y encuesta, además del riesgo de estigmatización territorial.

In [7]:
pd.read_csv(ROOT / 'outputs' / 'tables' / 'classification_permutation_importance.csv').head(10)

,feature,importance_mean,importance_std
0,victimizacion_robo_pct_lag1,0.100406,0.046006
1,victimizacion_pct_lag1,0.067555,0.041356
2,denuncias_total_lag1,0.058827,0.019891
3,percepcion_inseguridad_pct_lag1,0.040361,0.020435
4,prop_robo,0.039139,0.020929
5,denuncias_tasa_yoy,0.036290,0.000000
6,prop_hurto,0.026633,0.022961
7,confianza_pnp_pct_lag1,0.023201,0.008769
8,denuncias_tasa_100k_lag1,0.018978,0.021645
9,prop_violencia_contra_la_mujer_e_integrantes,0.016572,0.012837
